# Extracción, filtrado y clasificación BDI del corpus de Reddit

Este notebook reúne el pipeline completo en el orden en que se ejecutó: extracción mensual de submissions desde los volcados `.zst`, filtrado de los parquets, clasificación BDI contra el servidor local, validación del detector con eRisk 2024 y una auditoría final del parquet consolidado.

Los outputs de las secciones 1 a 4 se copiaron sin cambios de los notebooks originales. No se generaron al abrir este archivo: son la evidencia de la corrida que produjo los datos actuales. Las celdas nuevas (imports comunes, verificación corregida y auditoría) no tienen outputs.


## 1. Extracción de submissions desde `.zst`

Cada archivo mensual se lee en streaming con `zstandard`, sin descomprimir a disco ni pasar por un parquet intermedio. Se conservan solo los subreddits configurados y se escribe un parquet por mes con las 25 columnas del esquema común.

La ejecución es reanudable: si el parquet de salida ya existe, el mes se omite. Todos los imports del notebook están en la primera celda de esta sección.


In [1]:
# Imports comunes a las cinco secciones. Se declaran una sola vez.
import hashlib
import http.client
import io
import json
import os
import re
import threading
import time
from collections import Counter
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import zstandard as zstd
from tqdm.auto import tqdm


### Esquema de salida

`COLUMNS` fija el orden de las 25 columnas. `SCHEMA` fija los tipos; los campos que solo existen en comentarios (`link_id`, `parent_id`, `is_submitter`, `controversiality`) se escriben como nulos en las submissions.


In [5]:
COLUMNS = [
    "type", "id", "author_id", "author_deleted", "author", "created_utc", "created_year",
    "author_created_utc", "subreddit", "title", "text", "is_self", "link_id", "parent_id",
    "is_submitter", "score", "upvote_ratio", "num_comments", "gilded", "total_awards_received",
    "controversiality", "author_flair_text", "link_flair_text", "image_count", "image_urls",
]

SCHEMA = pa.schema([
    ("type",                  pa.string()),
    ("id",                    pa.string()),
    ("author_id",             pa.string()),
    ("author_deleted",        pa.int8()),
    ("author",                pa.string()),
    ("created_utc",           pa.int64()),
    ("created_year",          pa.int32()),
    ("author_created_utc",    pa.int64()),
    ("subreddit",             pa.string()),
    ("title",                 pa.string()),
    ("text",                  pa.string()),
    ("is_self",               pa.bool_()),
    ("link_id",               pa.string()),
    ("parent_id",             pa.string()),
    ("is_submitter",          pa.bool_()),
    ("score",                 pa.int64()),
    ("upvote_ratio",          pa.float64()),
    ("num_comments",          pa.int64()),
    ("gilded",                pa.int64()),
    ("total_awards_received", pa.int64()),
    ("controversiality",      pa.int64()),
    ("author_flair_text",     pa.string()),
    ("link_flair_text",       pa.string()),
    ("image_count",           pa.int32()),
    ("image_urls",            pa.list_(pa.string())),
])

### Helpers de normalización

Conversiones tolerantes a nulos y extracción de URLs de imágenes estáticas. `extract_static_image_urls` descarta vídeos, galerías y enlaces a GIF, y deduplica las URLs encontradas en el cuerpo del post.


In [6]:
def anon(user, salt=""):
    """Hash estable de 12 caracteres. None si falta o está borrado."""
    if user in (None, "", "[deleted]"): return None
    return hashlib.sha256((salt + user).encode("utf-8")).hexdigest()[:12]

def is_deleted(user):
    return 1 if user == "[deleted]" else 0

def ts_to_year(ts):
    if ts is None: return None
    try:
        return datetime.fromtimestamp(int(ts), tz=timezone.utc).year
    except (ValueError, TypeError, OSError):
        return None

def to_int(x):
    """int() tolerante: 1764547200.0 -> 1764547200, None/'' -> None."""
    try:
        return int(x)
    except (ValueError, TypeError):
        return None

def to_float(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return None

def to_str(x):
    return x if isinstance(x, str) else None

def to_bool(x):
    return x if isinstance(x, bool) else None

STATIC_EXTS = r'(?:jpg|jpeg|png)'
STATIC_IMAGE_RE = re.compile(
    r'https?://'
    r'(?:i\.redd\.it/|preview\.redd\.it/|external-preview\.redd\.it/|i\.imgur\.com/)'
    r'[^\s\)\]\}"\']+\.' + STATIC_EXTS,
    re.I,
)
IMGUR_BARE_RE  = re.compile(r'https?://(?:i\.)?imgur\.com/([A-Za-z0-9]+)(?![./\w-])', re.I)
IMGUR_ALBUM_RE = re.compile(r'https?://imgur\.com/(?:a|gallery)/', re.I)
GIF_HOST_RE    = re.compile(r'(giphy\.com|tenor\.com|gfycat\.com)', re.I)

def extract_static_image_urls(item):
    """URLs directas de imágenes estáticas (jpg/jpeg/png) de un submission."""
    urls = []

    if item.get("is_video") or item.get("is_gallery"):
        return []
    if item.get("post_hint") in ("rich:video", "hosted:video"):
        return []
    url = item.get("url") or ""
    if url and not GIF_HOST_RE.search(url) and STATIC_IMAGE_RE.match(url):
        urls.append(url.rstrip('.,;:!?'))

    text = item.get("selftext") or ""
    if isinstance(text, str) and text:
        for m in STATIC_IMAGE_RE.finditer(text):
            urls.append(m.group(0).rstrip('.,;:!?'))
        for m in IMGUR_BARE_RE.finditer(text):
            if IMGUR_ALBUM_RE.match(m.group(0)):
                continue
            urls.append(f"https://i.imgur.com/{m.group(1)}.jpg")

    seen, out = set(), []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

### Función principal

`process_zst` procesa el archivo por lotes de 100 000 filas y escribe primero un `.tmp`. Solo renombra el archivo al terminar, de modo que una interrupción no deja un parquet a medias.


In [7]:
def process_zst(in_path, out_path, subreddits=None, salt="", batch_size=100_000):
    """
    Lee un .zst de submissions (jsonl comprimido) en streaming, opcionalmente filtra
    por subreddit y escribe un parquet con exactamente las columnas de COLUMNS.
    No descomprime a disco. Devuelve (posts_leidos, posts_escritos).
    """
    wanted = None
    if subreddits is not None:
        wanted = {s.lower() for s in subreddits}

    rows = {c: [] for c in COLUMNS}
    total, written = 0, 0
    writer = pq.ParquetWriter(out_path + ".tmp", SCHEMA, compression="zstd")

    with open(in_path, "rb") as fh:
        # max_window_size alto: los dumps de Reddit usan ventanas de 2GB
        reader = zstd.ZstdDecompressor(max_window_size=2**31).stream_reader(fh)
        lines = io.TextIOWrapper(reader, encoding="utf-8", errors="replace")

        for line in tqdm(lines, desc=os.path.basename(in_path), unit=" lines", mininterval=5):
            total += 1
            try:
                post = json.loads(line)
            except json.JSONDecodeError:
                continue

            # filtro por subreddit
            if wanted is not None and (post.get("subreddit") or "").lower() not in wanted:
                continue

            author = post.get("author")
            imgs = extract_static_image_urls(post)
            created = post.get("created_utc")

            rows["type"].append("submission")
            rows["id"].append(to_str(post.get("id")))
            rows["author_id"].append(anon(author, salt))
            rows["author_deleted"].append(is_deleted(author))
            rows["author"].append(to_str(author))
            rows["created_utc"].append(to_int(created))
            rows["created_year"].append(ts_to_year(created))
            rows["author_created_utc"].append(to_int(post.get("author_created_utc")))
            rows["subreddit"].append(to_str(post.get("subreddit")))
            rows["title"].append(to_str(post.get("title")))
            rows["text"].append(to_str(post.get("selftext")))
            rows["is_self"].append(to_bool(post.get("is_self")))
            rows["link_id"].append(None)          # solo comentarios
            rows["parent_id"].append(None)        # solo comentarios
            rows["is_submitter"].append(None)     # solo comentarios
            rows["score"].append(to_int(post.get("score")))
            rows["upvote_ratio"].append(to_float(post.get("upvote_ratio")))
            rows["num_comments"].append(to_int(post.get("num_comments")))
            rows["gilded"].append(to_int(post.get("gilded")))
            rows["total_awards_received"].append(to_int(post.get("total_awards_received")))
            rows["controversiality"].append(None) # solo comentarios
            rows["author_flair_text"].append(to_str(post.get("author_flair_text")))
            rows["link_flair_text"].append(to_str(post.get("link_flair_text")))
            rows["image_count"].append(len(imgs))
            rows["image_urls"].append(imgs)
            written += 1

            # volcar lote a disco y vaciar listas
            if len(rows["id"]) >= batch_size:
                writer.write_table(pa.Table.from_pydict(rows, schema=SCHEMA))
                rows = {c: [] for c in COLUMNS}

    if rows["id"]:
        writer.write_table(pa.Table.from_pydict(rows, schema=SCHEMA))
    writer.close()
    os.replace(out_path + ".tmp", out_path)   # renombrar solo si terminó bien
    return total, written

### Configuración

`SUBREDDITS = None` desactiva el filtro. Los nombres se comparan en minúsculas y el `SALT` se usa para el hash estable de `author_id`.


In [8]:
INPUT_DIR  = "submissions"     # RS_2005-06.zst ... RS_2025-12.zst (submissions)
OUTPUT_DIR = "parquets_directos_reddit"    # un parquet por mes con las 25 columnas
SALT       = ""

SUBREDDITS = ["trans", "asktransgender", "MtF", "FtM", "NonBinary", "Transmasc", "TransLater"]  # None = sin filtro

os.makedirs(OUTPUT_DIR, exist_ok=True)

### Ejecución

Recorre `INPUT_DIR` en orden y salta los meses que ya tienen parquet. Los 493 outputs de esta celda se conservaron de la corrida original.


In [ ]:
for filename in sorted(os.listdir(INPUT_DIR)):
    if not filename.endswith(".zst"):
        continue
    out_path = os.path.join(OUTPUT_DIR, filename.replace(".zst", ".parquet"))
    if os.path.exists(out_path):
        continue
    total, kept = process_zst(os.path.join(INPUT_DIR, filename), out_path, subreddits=SUBREDDITS, salt=SALT)
    print(f"{filename}: {kept:,} de {total:,} posts")

RS_2005-06.zst: 0 lines [00:00, ? lines/s]

RS_2005-06.zst: 0 de 103 posts


RS_2005-07.zst: 0 lines [00:00, ? lines/s]

RS_2005-07.zst: 0 de 1,079 posts


RS_2005-08.zst: 0 lines [00:00, ? lines/s]

RS_2005-08.zst: 0 de 2,069 posts


... (483 outputs left out)

RS_2025-10.zst: 23,124 de 43,925,052 posts


RS_2025-11.zst: 0 lines [00:00, ? lines/s]

RS_2025-11.zst: 22,926 de 42,579,985 posts


RS_2025-12.zst: 0 lines [00:00, ? lines/s]

### Revisión

Conteo por subreddit del parquet completo. El output también corresponde a la corrida original.


In [18]:
df = pd.read_parquet(OUTPUT_DIR)
print(df.shape)
df.groupby("subreddit").size().sort_values(ascending=False)

(2544830, 25)


subreddit
trans             627130
asktransgender    561696
ftm               558181
MtF               448110
NonBinary         232717
TransLater         82636
TransMasc          34360
dtype: int64

## 2. Filtrado de parquets

Aplica sobre `parquet_filtrados/` los filtros que reducen el corpus antes de clasificar: descarta posts con imágenes, textos con `[deleted]` o `[removed]` y textos vacíos o demasiado cortos. El filtrado se hace archivo por archivo y conserva el nombre original.

La comparación final es opcional y sirve para inspeccionar un mes concreto.


In [ ]:
# ── Carpetas y archivos ────────────────────────────────────
CARPETA_ENTRADA = Path("parquets_directos_reddit")
CARPETA_SALIDA  = Path("parquet_filtrados")
PATRON_ARCHIVOS = "*.parquet"     # usa "**/*.parquet" para incluir subcarpetas
EXCLUIR_NOMBRES = []              # ej. ["tmp"] omite archivos cuyo nombre contenga "tmp"
SOBRESCRIBIR    = True            # False = omite archivos que ya existen en la salida

# ── Filtro al leer (formato pyarrow) ───────────────────────
FILTROS_LECTURA = [("image_count", "==", 0)]   # [] para no filtrar al leer

# ── Filtros de texto por fila ──────────────────────────────
COLUMNAS_A_REVISAR  = ["title", "text"]   # se descarta la fila si alguna contiene un patrón
PATRONES_PROHIBIDOS = ["[deleted]"]       # texto literal, sin distinguir mayúsculas
COLUMNAS_NO_VACIAS  = ["text"]            # se descarta la fila si está nula/vacía/solo espacios

### Funciones

`listar_archivos` resuelve la lista de entrada, `filtrar` aplica las condiciones por fila y `procesar` lee, filtra y escribe el parquet.


In [ ]:
def listar_archivos():
    """Devuelve los parquets de entrada, sin los excluidos."""
    archivos = sorted(CARPETA_ENTRADA.glob(PATRON_ARCHIVOS))
    return [a for a in archivos if not any(x in a.name for x in EXCLUIR_NOMBRES)]


def filtrar(df):
    """Aplica los filtros de texto y devuelve solo las filas que se conservan."""
    mantener = pd.Series(True, index=df.index)

    if PATRONES_PROHIBIDOS:
        regex = "|".join(re.escape(p) for p in PATRONES_PROHIBIDOS)
        for col in COLUMNAS_A_REVISAR:
            mantener &= ~df[col].str.contains(regex, case=False, na=False)

    for col in COLUMNAS_NO_VACIAS:
        mantener &= df[col].fillna("").str.strip() != ""

    return df[mantener]


def procesar(archivo):
    """Lee, filtra y guarda un parquet. Devuelve el número de filas conservadas."""
    df = pd.read_parquet(archivo, filters=FILTROS_LECTURA or None)
    df = filtrar(df)
    df.to_parquet(CARPETA_SALIDA / archivo.name, index=False, compression="zstd")
    return len(df)

### Ejecución

El output de esta celda es el embudo completo de los 247 archivos: 1 811 452 filas conservadas antes de la clasificación BDI.


In [12]:
CARPETA_SALIDA.mkdir(exist_ok=True)
archivos = listar_archivos()
total = 0

for i, archivo in enumerate(archivos, 1):
    if not SOBRESCRIBIR and (CARPETA_SALIDA / archivo.name).exists():
        print(f"[{i}/{len(archivos)}] {archivo.name} (ya existe, se omite)")
        continue
    filas = procesar(archivo)
    total += filas
    print(f"[{i}/{len(archivos)}] {archivo.name} -> {filas:,} filas")

print(f"Total filas: {total:,}")

[1/247] RS_2005-06.parquet -> 0 filas
[2/247] RS_2005-07.parquet -> 0 filas
[3/247] RS_2005-08.parquet -> 0 filas
[4/247] RS_2005-09.parquet -> 0 filas
[5/247] RS_2005-10.parquet -> 0 filas
[6/247] RS_2005-11.parquet -> 0 filas
[7/247] RS_2005-12.parquet -> 0 filas
[8/247] RS_2006-01.parquet -> 0 filas
[9/247] RS_2006-02.parquet -> 0 filas
[10/247] RS_2006-03.parquet -> 0 filas
[11/247] RS_2006-04.parquet -> 0 filas
[12/247] RS_2006-05.parquet -> 0 filas
[13/247] RS_2006-06.parquet -> 0 filas
[14/247] RS_2006-07.parquet -> 0 filas
[15/247] RS_2006-08.parquet -> 0 filas
[16/247] RS_2006-09.parquet -> 0 filas
[17/247] RS_2006-10.parquet -> 0 filas
[18/247] RS_2006-11.parquet -> 0 filas
[19/247] RS_2006-12.parquet -> 0 filas
[20/247] RS_2007-01.parquet -> 0 filas
[21/247] RS_2007-02.parquet -> 0 filas
[22/247] RS_2007-03.parquet -> 0 filas
[23/247] RS_2007-04.parquet -> 0 filas
[24/247] RS_2007-05.parquet -> 0 filas
[25/247] RS_2007-06.parquet -> 0 filas
[26/247] RS_2007-07.parquet -> 0 f

### Comparación opcional

Compara un archivo original con su versión filtrada. `INDICE_A_COMPARAR` se refiere a la posición dentro de `archivos`.


In [13]:
INDICE_A_COMPARAR = 182   # posición en la lista `archivos`

archivo = archivos[INDICE_A_COMPARAR]
for etiqueta, ruta in [("ORIGINAL", archivo), ("FILTRADO", CARPETA_SALIDA / archivo.name)]:
    df = pd.read_parquet(ruta)
    print(f"=== {etiqueta}: {ruta} ===")
    print(f"Filas: {len(df):,}")
    print(df.head(), "\n")

=== ORIGINAL: parquets_directos_reddit\RS_2020-08.parquet ===
Filas: 22,011
         type      id     author_id  author_deleted          author  \
0  submission  i1ik6x  74140e31bfeb               0  drummachine_on   
1  submission  i1ikxk  dfce1f3563a8               0     gliitchyboz   
2  submission  i1ikzr  e421efa5bda5               0           tujax   
3  submission  i1ilpi  2b15e65268c4               0    EggyEngineer   
4  submission  i1in5z           NaN               1       [deleted]   

   created_utc  created_year  author_created_utc       subreddit  \
0   1596240323          2020        1.587742e+09           trans   
1   1596240399          2020        1.543995e+09             ftm   
2   1596240406          2020        1.497186e+09  asktransgender   
3   1596240488          2020        1.444244e+09       NonBinary   
4   1596240631          2020                 NaN             ftm   

                                               title  ... score  upvote_ratio  \
0  for 

## 3. Clasificador BDI y unión en `datos/bdi_total.parquet`

**Modelo de clasificación utilizado:** [`startlux-models/StartLux-Decision-v1.1-4B-AutoThink`](https://huggingface.co/startlux-models/StartLux-Decision-v1.1-4B-AutoThink), servido localmente por HTTP en el endpoint `/v1/systemone` (`think_mode="never"`).

**Comando usado para levantar el servidor en WSL (entorno conda `vllm`):**

```bash
startlux-decide serve   --model models/StartLux-Decision-v1.1-4B-AutoThink   --launch-vllm --gpus 0 --port 8889   --gpu-memory-utilization 0.7   --max-model-len 4096   --no-mtp --no-images
```

**Versiones de software y hardware registradas:**
- `startlux-decide 1.1.0` · `vllm 0.33.0+rocm100`
- `Python 3.12.16` · `torch 2.13.0+rocm10.0.0` (HIP `7.15.26333`, `rocm-sdk 10.0.0`, `/opt/rocm-7.2.0`)
- `transformers 5.20.2` · `pyarrow 26.0.1` · `pandas 3.1.1` · `zstandard 0.25.0`
- GPU: `AMD Radeon RX 9070 XT` (16 GB VRAM, `gfx1201`)

---

Toma `parquet_filtrados/`, aplica un último filtro (bots, textos demasiado cortos) y consulta ese servidor local con una pregunta combinada por texto. La caché `bdi_resultados.jsonl` permite cortar y reanudar sin repetir consultas.

Cada parquet de `parquet_bdi/` conserva las columnas originales y añade `bdi` (`int8`) y `bdi_prob` (`float32`). La unión final deja en `datos/bdi_total.parquet` las 25 columnas originales más `bdi_prob`; la etiqueta binaria `bdi` no se incluye en el total.

Los outputs de las celdas de escaneo, prueba del cliente, clasificación, escritura y unión se conservan de la ejecución original.


In [ ]:
# ── Configuración ───────────────────────────────────────────
IN_DIR    = Path("parquet_filtrados")      # salida del notebook anterior
OUT_DIR   = Path("parquet_bdi")            # aquí quedan los parquets con la columna "bdi"
RESULTS   = Path("bdi_resultados.jsonl")   # caché reanudable {k: hash del texto, p: probabilidad}

HOST, PORT = "127.0.0.1", 8889
WORKERS    = 32                            # hilos concurrentes (sube/baja según el GPU)

MIN_CHARS  = 15                            # textos más cortos (tras limpiar URLs) se descartan
MAX_CHARS  = 3000                          # truncado (~750 tokens) para caber en max-model-len 4096
BOT_AUTHORS = {"automoderator"}            # autores a descartar (minúsculas)
BAD_PATTERN = r"\[(?:deleted|removed)\]"     # posts borrados/removidos
THRESHOLD  = 0.5                           # p >= THRESHOLD  ->  bdi = 1
CONF_COL = "bdi_prob"          # columna float32 con la probabilidad cruda del modelo
PERMITIR_INCOMPLETOS = False   # True = escribir bdi/bdi_prob nulos donde falte el resultado
FILES_LIMIT = None                         # p.ej. 3 para probar con pocos archivos; None = todos

QUESTION = (
    "Does the author show that they themselves are affected by any depression symptom, such as: "
    "sadness, hopelessness, feeling like a failure, loss of pleasure or interest, guilt, self-dislike, "
    "worthlessness, suicidal thoughts, crying, restlessness, indecisiveness, low energy, sleep or "
    "appetite changes, irritability, poor concentration, or fatigue?"
)

archivos = sorted(IN_DIR.glob("*.parquet"))
if FILES_LIMIT:
    archivos = archivos[:FILES_LIMIT]
OUT_DIR.mkdir(exist_ok=True)
print(f"{len(archivos)} archivos en {IN_DIR}/")

### Filtros y utilidades

`prepare` aplica el embudo de filtrado y devuelve la tabla recortada y los textos normalizados. `key_of` calcula la clave SHA-1 que identifica cada texto en la caché. `encolar` llena la cola de futuros y se comparte con la validación de eRisk para no duplicar la función.


In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.I)
WS_RE  = re.compile(r"\s+")

def _col(df, name):
    return df[name].fillna("").astype(str) if name in df.columns else pd.Series("", index=df.index)

def prepare(path):
    """Lee el parquet, aplica los filtros y devuelve (tabla_filtrada, textos, embudo)."""
    t = pq.read_table(path)
    names = t.schema.names
    df = t.select([c for c in ("title", "text", "author", "image_count") if c in names]).to_pandas()
    n0 = len(df)

    title, text, author = _col(df, "title"), _col(df, "text"), _col(df, "author")
    keep = pd.Series(True, index=df.index)
    funnel = {"entrada": n0}

    if "image_count" in df.columns:
        keep &= (df["image_count"].fillna(0) == 0)
    funnel["sin imágenes"] = int(keep.sum())

    bad = title.str.contains(BAD_PATTERN, case=False, regex=True) | text.str.contains(BAD_PATTERN, case=False, regex=True)
    keep &= ~bad
    funnel["sin [deleted]/[removed]"] = int(keep.sum())

    keep &= ~author.str.lower().isin(BOT_AUTHORS)
    funnel["sin bots"] = int(keep.sum())

    full = (title + "\n\n" + text).str.replace(URL_RE, " ", regex=True).str.replace(WS_RE, " ", regex=True).str.strip()
    keep &= full.str.len() >= MIN_CHARS
    funnel["texto >= MIN_CHARS"] = int(keep.sum())

    mask = keep.to_numpy()
    tabla = t.filter(pa.array(mask))
    textos = full[keep].str.slice(0, MAX_CHARS).tolist()
    return tabla, textos, funnel

def key_of(s):
    return hashlib.sha1(s.encode("utf-8")).hexdigest()


def encolar(pool, it, en_vuelo, ventana):
    # Llena la cola de futuros hasta el límite de la ventana.
    for k, s in it:
        en_vuelo[pool.submit(ask, s)] = k
        if len(en_vuelo) >= ventana:
            break


### Escaneo previo

Cuenta filas y textos únicos antes de consultar el servidor. El output muestra el embudo y las 1 550 027 consultas únicas de la corrida original.


In [7]:
unique = {}                 # hash -> texto
embudo_total = Counter()
filas_por_archivo = {}

for i, a in enumerate(archivos, 1):
    tabla, textos, f = prepare(a)
    embudo_total.update(f)
    filas_por_archivo[a.name] = tabla.num_rows
    for s in textos:
        unique.setdefault(key_of(s), s)
    if i % 20 == 0 or i == len(archivos):
        print(f"[{i}/{len(archivos)}] acumulado filas={sum(filas_por_archivo.values()):,}  textos únicos={len(unique):,}")

print("\nEmbudo de filtrado:")
prev = None
for k, v in embudo_total.items():
    print(f"  {k:<28}{v:>10,}" + (f"   (-{prev - v:,})" if prev is not None else ""))
    prev = v
print(f"\nConsultas a hacer: {len(unique):,}")

[20/247] acumulado filas=0  textos únicos=0
[40/247] acumulado filas=0  textos únicos=0
[60/247] acumulado filas=2  textos únicos=2
[80/247] acumulado filas=2,913  textos únicos=2,902
[100/247] acumulado filas=16,908  textos únicos=16,874
[120/247] acumulado filas=56,367  textos únicos=56,271
[140/247] acumulado filas=132,431  textos únicos=132,140
[160/247] acumulado filas=244,973  textos únicos=244,280
[180/247] acumulado filas=404,077  textos únicos=402,571
[200/247] acumulado filas=646,449  textos únicos=642,698
[220/247] acumulado filas=997,272  textos únicos=988,155
[240/247] acumulado filas=1,441,416  textos únicos=1,421,936
[247/247] acumulado filas=1,572,027  textos únicos=1,550,027

Embudo de filtrado:
  entrada                      1,811,452
  sin imágenes                 1,811,452   (-0)
  sin [deleted]/[removed]      1,579,139   (-232,313)
  sin bots                     1,572,335   (-6,804)
  texto >= MIN_CHARS           1,572,027   (-308)

Consultas a hacer: 1,550,027


### Cliente HTTP

Cada hilo mantiene su propia conexión persistente. `ask` reintenta con espera exponencial y devuelve `None` cuando se agotan los intentos, para que la clasificación pueda continuar y reintentar en la siguiente ejecución.


In [9]:
_tl = threading.local()

def _conn():
    c = getattr(_tl, "c", None)
    if c is None:
        c = _tl.c = http.client.HTTPConnection(HOST, PORT, timeout=120)
    return c

def ask(text, retries=5):
    body = json.dumps({
        "state": {"post": text},
        "think_mode": "never",
        "questions": {"any": {"type": "noul", "instructions": QUESTION}},
    })
    for n in range(retries):
        try:
            c = _conn()
            c.request("POST", "/v1/systemone", body, {"Content-Type": "application/json"})
            r = c.getresponse()
            data = r.read()
            if r.status != 200:
                raise RuntimeError(f"HTTP {r.status}: {data[:200]!r}")
            return float(json.loads(data)["answers"]["any"]["noul"])
        except Exception as e:
            _tl.c = None
            if n == retries - 1:
                return None
            time.sleep(0.5 * 2 ** n)

# prueba con 5 textos
muestra = list(unique.items())[:5]
t0 = time.time()
for k, s in muestra:
    print(f"{ask(s)!s:>22}  {s[:90]!r}")
print(f"{(time.time()-t0)/max(len(muestra),1)*1000:.0f} ms/consulta (secuencial)")

  0.011965784090016663  "What's on your mind? This is the place to get some answers. This reddit, inspired by askRe"
   0.06844012530803281  'Is tranny the new "N-Word"? I will admit I say it among my friends (being trans and "ownin'
   0.01625694504311803  "Does anyone have any fitness advice for someone wanting to look more feminine? I'm startin"
   0.09639697705577398  'Better Transman Guides? Hello. While I realize there are a good number of transman guides,'
   0.11390874268450969  'Dear Internet: Thank you I spend several hours each day researching the Internet for all a'
82 ms/consulta (secuencial)


### Clasificación reanudable

Los imports de `ThreadPoolExecutor`, `wait`, `FIRST_COMPLETED`, `tqdm` y `datetime` están en la primera celda del notebook; esta celda solo conserva la lógica de la cola y el volcado periódico de la caché. Si se interrumpe, vuelve a ejecutarse y continúa desde `bdi_resultados.jsonl`.


In [10]:
# ── Cargar caché (NO se reprocesa nada de lo ya guardado) ──
probs = {}
if RESULTS.exists():
    with open(RESULTS, encoding="utf-8") as f:
        for line in f:
            try:
                d = json.loads(line)
                probs[d["k"]] = d["p"]
            except Exception:
                pass            # línea truncada por una interrupción

pendientes = [(k, s) for k, s in unique.items() if k not in probs]
print(f"Ya hechos: {len(probs):,} · pendientes: {len(pendientes):,}")

WINDOW = WORKERS * 4            # peticiones en vuelo a la vez
FLUSH_EVERY = 5                 # segundos entre flush a disco
fallos = hechos = 0
total = len(pendientes)
t0 = last_flush = time.time()

def fmt_eta(seg):
    return str(timedelta(seconds=int(seg)))

pool = ThreadPoolExecutor(WORKERS)
out = open(RESULTS, "a", encoding="utf-8")
pbar = tqdm(total=total, unit="req", dynamic_ncols=True, smoothing=0.05, mininterval=0.5)
en_vuelo = {}
it = iter(pendientes)

try:
    encolar(pool, it, en_vuelo, WINDOW)
    while en_vuelo:
        done, _ = wait(en_vuelo, return_when=FIRST_COMPLETED)
        for fut in done:
            k = en_vuelo.pop(fut)
            p = fut.result()
            if p is None:
                fallos += 1
            else:
                probs[k] = p
                out.write(json.dumps({"k": k, "p": p}) + "\n")
            hechos += 1
            pbar.update(1)

        now = time.time()
        el = now - t0
        rate = hechos / el if el > 0 else 0
        rest = (total - hechos) / rate if rate > 0 else 0
        pbar.set_postfix(
            fallos=fallos,
            rate=f"{rate:.1f}/s",
            faltan=f"{total - hechos:,}",
            ETA=fmt_eta(rest),
            fin=(datetime.now() + timedelta(seconds=rest)).strftime("%a %H:%M"),
        )
        if now - last_flush >= FLUSH_EVERY:
            out.flush()
            last_flush = now
        encolar(pool, it, en_vuelo, WINDOW)
except KeyboardInterrupt:
    print("\nInterrumpido: guardando lo procesado...")
    for fut in en_vuelo:
        fut.cancel()
finally:
    out.flush(); out.close()
    pbar.close()
    pool.shutdown(wait=True, cancel_futures=True)

print(f"\nListo. Con resultado: {len(probs):,}/{len(unique):,} · fallos (se reintentan al re-ejecutar): {fallos}")

Ya hechos: 1,550,026 · pendientes: 1


  0%|          | 0/1 [00:00<?, ?req/s]


Listo. Con resultado: 1,550,026/1,550,027 · fallos (se reintentan al re-ejecutar): 1


### Escritura de parquets etiquetados

Solo se escribe un archivo cuando todas sus filas tienen resultado. Los incompletos se saltan y se reintentan al volver a ejecutar la clasificación. El output conserva el caso de `RS_2024-06`, que quedó fuera por dos resultados faltantes.


In [11]:
# Reconstruye las probabilidades del JSONL si el kernel es nuevo (no usa el modelo)
if "probs" not in globals() or not probs:
    probs = {}
    with open(RESULTS, encoding="utf-8") as f:
        for line in f:
            try:
                d = json.loads(line)
            except Exception:
                continue
            probs[d["k"]] = d["p"]
print(f"Probabilidades disponibles: {len(probs):,}")

resumen, saltados = [], 0
for i, a in enumerate(archivos, 1):
    tabla, textos, _ = prepare(a)
    ps = [probs.get(key_of(s)) for s in textos]
    nulos = sum(p is None for p in ps)

    if nulos and not PERMITIR_INCOMPLETOS:
        saltados += 1
        print(f"[{i}/{len(archivos)}] SALTADO (faltan {nulos:,} resultados): {a.name}")
        continue

    bdi  = pa.array([None if p is None else int(p >= THRESHOLD) for p in ps], type=pa.int8())
    conf = pa.array(ps, type=pa.float32())
    salida = tabla.append_column("bdi", bdi).append_column(CONF_COL, conf)

    tmp = OUT_DIR / (a.name + ".tmp")
    pq.write_table(salida, tmp)
    tmp.replace(OUT_DIR / a.name)

    resumen.append((a.name, tabla.num_rows, sum(1 for p in ps if p is not None and p >= THRESHOLD)))
    if i % 20 == 0 or i == len(archivos):
        print(f"[{i}/{len(archivos)}] escritos")

tot = sum(r[1] for r in resumen); pos = sum(r[2] for r in resumen)
print(f"\nArchivos escritos: {len(resumen)} · filas: {tot:,} · bdi=1: {pos:,} ({pos/max(tot,1):.1%}) · bdi=0: {tot-pos:,}")
print(f"Columnas añadidas: bdi (int8, THRESHOLD={THRESHOLD}) + {CONF_COL} (float32)")
if saltados:
    print(f"{saltados} archivo(s) saltado(s): re-ejecuta la clasificación de la sección 3 cuando el servidor esté arriba.")

Probabilidades disponibles: 1,550,026
[20/247] escritos
[40/247] escritos
[60/247] escritos
[80/247] escritos
[100/247] escritos
[120/247] escritos
[140/247] escritos
[160/247] escritos
[180/247] escritos
[200/247] escritos
[220/247] escritos
[229/247] SALTADO (faltan 2 resultados): RS_2024-06.parquet
[240/247] escritos
[247/247] escritos

Archivos escritos: 246 · filas: 1,548,936 · bdi=1: 492,772 (31.8%) · bdi=0: 1,056,164
Columnas añadidas: bdi (int8, THRESHOLD=0.5) + bdi_prob (float32)
1 archivo(s) saltado(s): re-ejecuta la celda 3 cuando el servidor esté arriba.


### Verificación del esquema

Comprueba que cada archivo tenga las columnas originales más `bdi` y `bdi_prob`, que los tipos sean los esperados y que la etiqueta binaria coincida con `bdi_prob >= THRESHOLD`. La versión original solo aceptaba `bdi`, por eso marcaba los 246 archivos como distintos.


In [ ]:
errores = 0
for a in archivos:
    dst = OUT_DIR / a.name
    if not dst.exists():
        continue

    s_in, s_out = pq.read_schema(a), pq.read_schema(dst)
    extras = {"bdi": pa.int8(), CONF_COL: pa.float32()}

    ok = s_out.names == s_in.names + [c for c in extras if c in s_out.names]
    ok = ok and all(c in s_out.names for c in extras)
    ok = ok and all(s_out.field(n).type == s_in.field(n).type for n in s_in.names)
    ok = ok and all(s_out.field(c).type == t for c, t in extras.items())

    tabla = pq.read_table(dst, columns=["bdi", CONF_COL])
    bdi = tabla.column("bdi").to_numpy(zero_copy_only=False)
    conf = tabla.column(CONF_COL).to_numpy(zero_copy_only=False)
    bdi_validos = bdi[~pd.isna(bdi)]
    conf_validos = conf[~pd.isna(conf)]

    ok = ok and set(np.unique(bdi_validos)) <= {0, 1}
    ok = ok and bool(((conf_validos >= 0) & (conf_validos <= 1)).all())
    ok = ok and bool((pd.isna(conf) | (bdi == (conf >= THRESHOLD))).all())

    if not ok:
        errores += 1
        print("ESQUEMA DISTINTO:", a.name)

print("Esquemas OK" if errores == 0 else f"{errores} archivos con problemas")

ej = next((OUT_DIR / a.name for a in archivos if (OUT_DIR / a.name).exists()), None)
if ej is not None:
    df = pd.read_parquet(ej)
    print(ej.name, df.shape)
    display(df[[c for c in ("title", "bdi", CONF_COL) if c in df.columns]].head(10))


### Unión en `datos/bdi_total.parquet`

Toma el esquema de un archivo con datos, fija los tipos que varían entre meses y escribe la unión sin la columna `bdi`. El resultado tiene 26 columnas y 1 548 936 filas.


In [14]:
UNIR = True
if UNIR:
    destino = Path("datos/bdi_total.parquet")
    destino.parent.mkdir(parents=True, exist_ok=True)
    existentes = [OUT_DIR / a.name for a in archivos if (OUT_DIR / a.name).exists()]

    # Esquema unificado: tomamos el de un archivo con datos reales y fijamos
    # los tipos que varían entre archivos.
    base = pq.read_schema(max(existentes, key=lambda f: pq.ParquetFile(f).metadata.num_rows))
    overrides = {
        "author_created_utc":    pa.float64(),   # acepta int y NaN
        "total_awards_received": pa.float64(),
        "controversiality":      pa.float64(),
        "is_submitter":          pa.bool_(),
        "image_urls":            pa.list_(pa.string()),
    }

    # bdi_total se queda SOLO con la confianza; la etiqueta binaria se excluye.
    EXCLUIR = {"bdi"}
    if CONF_COL not in base.names:
        raise SystemExit(f"'{CONF_COL}' no está en los parquets: ejecuta antes la escritura de la sección 3.")

    fields = [pa.field(f.name, overrides.get(f.name, f.type))
              for f in base if f.name not in EXCLUIR]
    schema = pa.schema(fields)

    writer = pq.ParquetWriter(destino, schema)
    total = 0
    for f in existentes:
        t = pq.read_table(f)
        t = t.select(schema.names).cast(schema, safe=False)   # 'bdi' no está en schema.names
        writer.write_table(t)
        total += t.num_rows
    writer.close()
    print(f"Escrito {destino.name} · {len(existentes)} archivos · {total:,} filas · con {CONF_COL}, sin bdi")

Escrito bdi_total.parquet · 246 archivos · 1,548,936 filas · con bdi_prob, sin bdi


## 4. Validación del detector en eRisk 2024

Usa los textos anotados de `datos/eRisk_2024_dataset_majority.csv` y la caché `datos/erisk_resultados.jsonl`. El grupo `positivo` tiene al menos un síntoma confirmado; el `control` reúne textos con cinco o más síntomas anotados y todos en cero.

La métrica principal es el recall del grupo positivo, no la precisión. El control sirve para detectar un detector que responde "sí" a casi todo. El cliente HTTP, la pregunta y los umbrales se reutilizan de la sección 3.


In [ ]:
# ── Configuración de la validación ─────────────────────────
CSV     = Path("datos/eRisk_2024_dataset_majority.csv")
RESULTS = Path("datos/erisk_resultados.jsonl")

INCLUIR_CONTROL     = True
MIN_QUERIES_CONTROL = 5

# HOST, PORT, WORKERS, THRESHOLD, MAX_CHARS y QUESTION se reutilizan
# de la sección 3 para que ambas evaluaciones usen el mismo criterio.
assert CSV.exists(), f"no existe {CSV}"
RESULTS.parent.mkdir(parents=True, exist_ok=True)


### Carga y normalización

Normaliza los nombres de columnas (`DOCNO`, `relevant`, `TEXT`), recorta los textos a `MAX_CHARS` y colapsa los pares síntoma-texto en una fila por texto.


In [2]:
# ── Cargar y normalizar (docid/doc_id/DOCNO, rel/relevant, text/TEXT) ──
df = pd.read_csv(CSV)
df = df.rename(columns={"DOCNO": "docid", "relevant": "rel", "TEXT": "text"})
df["rel"] = df["rel"].astype(int)
df["texto"] = df["text"].fillna("").astype(str).str.strip().str.slice(0, MAX_CHARS)
df = df[df["texto"].str.len() > 0]

print(f"archivo: {CSV}")
print(f"pares (síntoma, texto): {len(df):,} · textos únicos: {df['texto'].nunique():,}")

# ── Una fila por texto: ¿positivo o control? ───────────────
resumen = (df.groupby("texto", as_index=False)
             .agg(n_queries=("query", "nunique"), n_positivos=("rel", "sum")))

resumen["grupo"] = np.where(
    resumen["n_positivos"] > 0, "positivo",
    np.where((resumen["n_positivos"] == 0) & (resumen["n_queries"] >= MIN_QUERIES_CONTROL), "control", None),
)
if not INCLUIR_CONTROL:
    resumen.loc[resumen["grupo"] == "control", "grupo"] = None

bench = resumen.dropna(subset=["grupo"]).reset_index(drop=True)
print(bench["grupo"].value_counts().to_string())
print(f"textos a clasificar: {len(bench):,}")

archivo: eRisk_2024_dataset_majority.csv
pares (síntoma, texto): 14,813 · textos únicos: 10,733
grupo
positivo    5737
control      186
textos a clasificar: 5,923


### Prueba del cliente

Reutiliza `ask` de la sección 3; no se redefine el cliente. La prueba es secuencial y sirve para confirmar que el servidor responde antes de lanzar los 32 hilos.


In [3]:
# prueba con 5 textos del benchmark
muestra = bench["texto"].head(5).tolist()
t0 = time.time()
for s in muestra:
    print(f"{ask(s)!s:>22}  {s[:80]!r}")
print(f"{(time.time() - t0) / max(len(muestra), 1) * 1000:.0f} ms/consulta (secuencial)")

    0.9962965797305816  '"...."\n\n\nI feel like crying.'
    0.9856163255047442  '"I hate that I hate myself."'
    0.9597042794910917  '"I\'m tired, I have a lot of work that I still need to finish, I don\'t feel well,'
    0.9872765276465266  '"I\'m tired.'
    0.9792489162918528  '"No, I just feel bad that I let this happen."'
231 ms/consulta (secuencial)


### Clasificación reanudable

Misma cola que la sección 3, con `WINDOW = WORKERS * 4` y volcado a disco cada cinco segundos. La caché `datos/erisk_resultados.jsonl` evita repetir consultas.


In [4]:
# ── Cargar caché (NO se repite nada ya guardado) ───────────
probs = {}
if RESULTS.exists():
    with open(RESULTS, encoding="utf-8") as f:
        for line in f:
            try:
                d = json.loads(line)
                probs[d["k"]] = d["p"]
            except Exception:
                pass            # línea truncada por una interrupción

textos_unicos = list(dict.fromkeys(bench["texto"]))
pendientes = []
for s in textos_unicos:
    k = key_of(s)
    if k not in probs:
        pendientes.append((k, s))

print(f"Textos únicos: {len(textos_unicos):,} · ya hechos: {len(textos_unicos) - len(pendientes):,} · pendientes: {len(pendientes):,}")

WINDOW = WORKERS * 4            # peticiones en vuelo a la vez
FLUSH_EVERY = 5                 # segundos entre flush a disco
fallos = hechos = 0
total = len(pendientes)
t0 = last_flush = time.time()

pool = ThreadPoolExecutor(WORKERS)
out = open(RESULTS, "a", encoding="utf-8")
pbar = tqdm(total=total, unit="req", dynamic_ncols=True)
en_vuelo = {}
it = iter(pendientes)

try:
    encolar(pool, it, en_vuelo, WINDOW)
    while en_vuelo:
        done, _ = wait(en_vuelo, return_when=FIRST_COMPLETED)
        for fut in done:
            k = en_vuelo.pop(fut)
            p = fut.result()
            if p is None:
                fallos += 1
            else:
                probs[k] = p
                out.write(json.dumps({"k": k, "p": p}) + "\n")
            hechos += 1
            pbar.update(1)

        if time.time() - last_flush >= FLUSH_EVERY:
            out.flush()
            last_flush = time.time()
        encolar(pool, it, en_vuelo, WINDOW)
except KeyboardInterrupt:
    print("\nInterrumpido: guardando lo procesado...")
    for fut in en_vuelo:
        fut.cancel()
finally:
    out.flush(); out.close()
    pbar.close()
    pool.shutdown(wait=True, cancel_futures=True)

print(f"\nListo. Con resultado: {sum(1 for s in textos_unicos if key_of(s) in probs):,}/{len(textos_unicos):,} · fallos (se reintentan al re-ejecutar): {fallos}")

Textos únicos: 5,923 · ya hechos: 0 · pendientes: 5,923


  0%|          | 0/5923 [00:00<?, ?req/s]


Listo. Con resultado: 5,923/5,923 · fallos (se reintentan al re-ejecutar): 0


### Resultados globales

Resume la distribución de `bdi_prob` por grupo y el porcentaje que supera `THRESHOLD`. El output conserva la corrida original: 78,8 % de recall en positivo y 18,3 % de marca en el control.


In [5]:
# ── Pegar las probabilidades y resumir ─────────────────────
p_por_texto = {s: probs.get(key_of(s)) for s in textos_unicos}
bench["p"] = bench["texto"].map(p_por_texto)
bench["marca"] = (bench["p"] >= THRESHOLD)

faltan = int(bench["p"].isna().sum())
if faltan:
    print(f"AVISO: {faltan:,} textos sin resultado; re-ejecuta la celda de clasificación.\n")

tabla = bench.groupby("grupo")["p"].agg(
    n="size", media="mean", mediana="median",
    p10=lambda s: s.quantile(0.10), p90=lambda s: s.quantile(0.90),
)
tabla["% >= THRESHOLD"] = bench.groupby("grupo")["marca"].mean() * 100
print(tabla.round(3).to_string())

print("\n── Distribución de bdi_prob por grupo ──")
centiles = [(0.0, .1), (.1, .3), (.3, .5), (.5, .7), (.7, .9), (.9, 1.0001)]
for grupo in ["positivo", "control"]:
    sub = bench.loc[bench["grupo"] == grupo, "p"].dropna()
    if len(sub) == 0:
        continue
    print(f"\n{grupo} (n={len(sub):,})")
    for lo, hi in centiles:
        m = (sub >= lo) & (sub < hi)
        print(f"  [{lo:.1f}, {min(hi, 1.0):.1f}] {m.sum():>6,}  {100 * m.mean():5.1f}%")

             n  media  mediana    p10    p90  % >= THRESHOLD
grupo                                                       
control    186  0.249    0.127  0.020  0.765          18.280
positivo  5737  0.773    0.957  0.203  0.996          78.787

── Distribución de bdi_prob por grupo ──

positivo (n=5,737)
  [0.0, 0.1]    376    6.6%
  [0.1, 0.3]    398    6.9%
  [0.3, 0.5]    443    7.7%
  [0.5, 0.7]    398    6.9%
  [0.7, 0.9]    672   11.7%
  [0.9, 1.0]  3,450   60.1%

control (n=186)
  [0.0, 0.1]     77   41.4%
  [0.1, 0.3]     55   29.6%
  [0.3, 0.5]     20   10.8%
  [0.5, 0.7]     12    6.5%
  [0.7, 0.9]     13    7.0%
  [0.9, 1.0]      9    4.8%


### Recall por síntoma

Desglosa el grupo positivo por `query`. El recall global se reporta de dos formas: macro (media de los 21 síntomas) y micro (media sobre todos los pares).


In [6]:
# ── Recall por síntoma (el grupo positivo desglosado) ──────
detalle = df.merge(bench[["texto", "p", "grupo"]], on="texto", how="inner")
pos = detalle[(detalle["grupo"] == "positivo") & (detalle["rel"] == 1)].copy()
pos["marca"] = (pos["p"] >= THRESHOLD)

por_query = pos.groupby("query").agg(
    textos=("p", "size"),
    prob_media=("p", "mean"),
    recall=("marca", "mean"),
)
por_query["recall"] = (por_query["recall"] * 100).round(1)
print(por_query.round(3).to_string())
print(f"\nRecall global (macro entre síntomas): {por_query['recall'].mean():.1f} %")
print(f"Recall global (micro, todos los pares): {pos['marca'].mean() * 100:.1f} %")

       textos  prob_media  recall
query                            
1         442       0.787    80.3
2         294       0.727    72.1
3         270       0.950    97.8
4         196       0.817    84.2
5         399       0.939    94.0
6         162       0.559    60.5
7         385       0.917    94.0
8         281       0.904    96.1
9         410       0.920    92.0
10        433       0.942    95.2
11        400       0.510    39.2
12        211       0.888    90.0
13        308       0.545    55.8
14        258       0.976    99.2
15        243       0.805    82.7
16        365       0.573    67.4
17        305       0.773    85.9
18        334       0.684    68.3
19        286       0.825    85.0
20        343       0.909    93.9
21        304       0.488    45.7

Recall global (macro entre síntomas): 80.0 %
Recall global (micro, todos los pares): 80.0 %


### Cómo leer los resultados

- `% >= THRESHOLD` del grupo positivo es el recall: de cada 100 textos anotados como "sí expresa un síntoma", cuántos marca el detector.
- El control debe mirarse al lado. Si también sale mayoritariamente positivo, el modelo está respondiendo "sí" a casi todo y el recall alto no prueba nada.
- La media y la mediana informan más que el umbral: una mediana alta en positivo y baja en control indica separación aunque el recall a 0.5 sea imperfecto.
- Los fragmentos de eRisk son cortos y van sin título; el detector se calibró con posts largos. Es un caso difícil por diseño, no una estimación del rendimiento sobre el corpus.
- Referencia del corpus: en `datos/bdi_total.parquet`, 31,8 % de las filas quedaron sobre 0,5 y 10,4 % en la banda 0,40–0,60.


## 5. Auditoría de `datos/bdi_total.parquet`

Auditoría de solo lectura, sin outputs ejecutados. Verifica la forma esperada, normaliza `subreddit` a minúsculas y resume el volumen y la proporción de `bdi_prob >= 0.5` por subreddit. También revisa el rango de años, la cobertura de `RS_2024-06` y la higiene de `author` y `author_id`.

Si abres el notebook solo para correr esta auditoría (sin volver a ejecutar las secciones 1 a 4), corre primero la celda de imports al inicio de la sección 1. La celda no escribe nada; si falla una comprobación básica, lanza `AssertionError`.


In [2]:
# ── Auditoría de solo lectura ────────────────────────────────
RUTA_BDI = Path("datos/bdi_total.parquet")
FORMA_ESPERADA = (1_548_936, 26)

assert RUTA_BDI.exists(), f"no existe {RUTA_BDI}"
pf = pq.ParquetFile(RUTA_BDI)
filas = pf.metadata.num_rows
columnas = len(pf.schema_arrow)
print(f"Archivo: {RUTA_BDI}")
print(f"Forma: {filas:,} x {columnas}")
assert (filas, columnas) == FORMA_ESPERADA, (
    f"se esperaba {FORMA_ESPERADA[0]:,} x {FORMA_ESPERADA[1]}, "
    f"se encontró {filas:,} x {columnas}"
)

requeridas = {"subreddit", "created_year", "created_utc", "bdi_prob",
              "author", "author_id", "author_deleted"}
faltantes = requeridas - set(pf.schema_arrow.names)
assert not faltantes, f"faltan columnas: {sorted(faltantes)}"

df_bdi = pf.read(columns=sorted(requeridas)).to_pandas()
assert len(df_bdi) == filas

# ── Subreddit: normalización y conteos ───────────────────────
df_bdi["subreddit"] = (
    df_bdi["subreddit"].fillna("(sin subreddit)").astype(str).str.strip().str.lower()
)
assert df_bdi["subreddit"].str.lower().eq(df_bdi["subreddit"]).all()
print(f"\nSubreddits distintos: {df_bdi['subreddit'].nunique():,}")

validos = int(df_bdi["bdi_prob"].notna().sum())
positivos = int((df_bdi["bdi_prob"] >= 0.5).sum())
assert len(df_bdi) == validos, f"bdi_prob tiene {len(df_bdi) - validos:,} nulos"
assert df_bdi["bdi_prob"].between(0, 1).all()
print(f"bdi_prob >= 0.5: {positivos:,} de {validos:,} "
      f"({100 * positivos / validos:.2f} %)")

resumen = (
    df_bdi.assign(bdi_positivo=df_bdi["bdi_prob"] >= 0.5)
    .groupby("subreddit", dropna=False)
    .agg(filas=("bdi_prob", "size"),
         positivos=("bdi_positivo", "sum"))
    .sort_values("filas", ascending=False)
)
resumen["pct"] = 100 * resumen["positivos"] / resumen["filas"]

print(f"\n{'subreddit':<22}{'filas':>12}{'bdi>=0.5':>12}{'%':>9}")
for sub, r in resumen.iterrows():
    print(f"{sub:<22}{int(r['filas']):>12,}{int(r['positivos']):>12,}"
          f"{r['pct']:>8.2f}%")

# ── Rango de años y cobertura mensual ───────────────────────
anios = df_bdi["created_year"].dropna().astype(int)
assert len(anios) > 0, "created_year está vacío"
print(f"\nRango de años: {anios.min()}–{anios.max()}")
print("Filas por año:")
for anio, n in df_bdi.groupby("created_year").size().items():
    print(f"  {int(anio):>4}  {n:>10,}")
print(f"  created_year nulo: {int(df_bdi['created_year'].isna().sum()):,}")

fechas = pd.to_datetime(df_bdi["created_utc"], unit="s", utc=True, errors="coerce")
meses = fechas.dt.strftime("%Y-%m")
junio_2024 = int((meses == "2024-06").sum())
meses_2024 = int(meses[df_bdi["created_year"].eq(2024)].nunique())
print(f"\nRS_2024-06: {junio_2024:,} filas · meses de 2024 con datos: {meses_2024}")
if junio_2024 == 0:
    print("  El parquet RS_2024-06 no está en bdi_total: el clasificador lo saltó")
    print("  porque faltaban 2 resultados (ver la escritura de la sección 3).")
else:
    print("  El parquet RS_2024-06 sí está incluido.")

# ── Higiene de author ────────────────────────────────────────
author_crudo = df_bdi["author"]
author = author_crudo.fillna("")
deleted = author.eq("[deleted]")
flag = df_bdi["author_deleted"].fillna(0).astype(int)
assert set(flag.unique()) <= {0, 1}

nulos_author = int(author_crudo.isna().sum())
vacios = int(author.str.strip().eq("").sum())
borrados = int(deleted.sum())
coincide = int((flag == deleted.astype(int)).sum())
aid_nulo = df_bdi["author_id"].isna()
huecos = int((aid_nulo & ~deleted).sum())

print("\nHigiene de author:")
print(f"  autores distintos: {int(author_crudo.nunique(dropna=True)):,}")
print(f"  author nulo: {nulos_author:,} · vacío: {vacios:,} · [deleted]: {borrados:,}")
print(f"  author_deleted=1: {int(flag.sum()):,} · "
      f"coincide con author == '[deleted]': {coincide:,}/{len(df_bdi):,}")
print(f"  author_id nulo: {int(aid_nulo.sum()):,} · "
      f"sin id pese a no estar borrado: {huecos:,}")

pares = df_bdi.loc[~deleted & author.ne("") & df_bdi["author_id"].notna(),
                   ["author", "author_id"]]
ids_por_autor = pares.groupby("author")["author_id"].nunique()
print(f"  autores con más de un author_id: {int((ids_por_autor > 1).sum()):,}")

muestra = pares.drop_duplicates("author").head(50_000)
esperado = muestra["author"].map(
    lambda s: hashlib.sha256(str(s).encode("utf-8")).hexdigest()[:12]
)
malos = int((esperado != muestra["author_id"]).sum())
print(f"  hashes comprobados: {len(muestra):,} · discrepancias: {malos:,}")
if malos:
    print("  AVISO: author_id no coincide con SHA-256[:12] sin sal en la muestra.")

top = author[~deleted & author.ne("")].value_counts().head(5)
print("  autores con más filas:")
for nombre, n in top.items():
    print(f"    {str(nombre)[:24]:<24}{n:>10,}")


Archivo: datos/bdi_total.parquet
Forma: 1,548,936 x 26

Subreddits distintos: 7
bdi_prob >= 0.5: 492,772 de 1,548,936 (31.81 %)

subreddit                    filas    bdi>=0.5        %
asktransgender             433,481     135,048   31.15%
ftm                        398,024     117,955   29.64%
mtf                        321,530     112,242   34.91%
trans                      265,606      90,833   34.20%
nonbinary                   80,025      21,216   26.51%
translater                  27,111       8,180   30.17%
transmasc                   23,159       7,298   31.51%

Rango de años: 2009–2025
Filas por año:
  2009           2
  2010         102
  2011       2,337
  2012       6,588
  2013      11,400
  2014      20,765
  2015      37,024
  2016      49,341
  2017      60,249
  2018      76,823
  2019      98,287
  2020     106,610
  2021     160,806
  2022     194,766
  2023     235,359
  2024     249,799
  2025     238,678
  created_year nulo: 0

RS_2024-06: 0 filas · meses de 2024